In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Deterministic and Bayesian Refinement: LBCO, HRPT

This tutorial demonstrates a practical two-stage workflow for powder
diffraction analysis with EasyDiffraction.

In the first stage, we run a fast local refinement to obtain a sensible
point estimate and parameter uncertainties. In the second stage, we use
these refined values to define fit bounds and then sample the posterior
distribution with DREAM.

The example uses constant-wavelength neutron powder diffraction data
for La0.5Ba0.5CoO3 measured on HRPT at PSI.

The goal is not only to obtain a good fit, but also to answer Bayesian
questions such as:

- Which parameter values are most probable?
- How broad are the credible intervals?
- Which parameters are strongly correlated?
- How much uncertainty propagates into the calculated diffraction
  pattern?

## Import Library

In [2]:
import easydiffraction as ed

## Step 1: Create a Project Container

The project object keeps structures, experiments, fit settings, and
plotting utilities together in a single place. We will build the full
workflow inside this object.

In [3]:
project = ed.Project()

## Step 2: Build the Structural Model

We define a simple cubic perovskite model for LBCO. La and Ba share the
same crystallographic site with equal occupancy, while Co and O occupy
the remaining ideal perovskite positions.

In [4]:
project.structures.create(name='lbco')

In [5]:
structure = project.structures['lbco']

In [6]:
structure.space_group.name_h_m = 'P m -3 m'
structure.space_group.it_coordinate_system_code = '1'

In [7]:
structure.cell.length_a = 3.88

The atom-site definitions below form the starting structural model. The
parameters are intentionally reasonable rather than fully optimized,
because the refinement step will improve them.

In [8]:
structure.atom_sites.create(
    label='La',
    type_symbol='La',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    wyckoff_letter='a',
    adp_type='Biso',
    adp_iso=0.5151,
    occupancy=0.5,
)
structure.atom_sites.create(
    label='Ba',
    type_symbol='Ba',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    wyckoff_letter='a',
    adp_type='Biso',
    adp_iso=0.5151,
    occupancy=0.5,
)
structure.atom_sites.create(
    label='Co',
    type_symbol='Co',
    fract_x=0.5,
    fract_y=0.5,
    fract_z=0.5,
    wyckoff_letter='b',
    adp_type='Biso',
    adp_iso=0.2190,
)
structure.atom_sites.create(
    label='O',
    type_symbol='O',
    fract_x=0,
    fract_y=0.5,
    fract_z=0.5,
    wyckoff_letter='c',
    adp_type='Biso',
    adp_iso=1.3916,
)

## Step 3: Define the Diffraction Experiment

Next we download the measured powder pattern, create a neutron powder
experiment, and configure the instrument, profile, background, and
excluded regions.

#### Download the Measured Data

In [9]:
data_path = ed.download_data(id=3, destination='data')

Getting data...
Data #3: La0.5Ba0.5CoO3, HRPT (PSI), 300 K
✅ Data #3 already present at 'data/ed-3.xye'. Keeping existing file.


#### Create the Experiment Object

In [10]:
project.experiments.add_from_data_path(
    name='hrpt',
    data_path=data_path,
    sample_form='powder',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
)

Data loaded successfully
Experiment 🔬 'hrpt'. Number of data points: 3098.


In [11]:
experiment = project.experiments['hrpt']

#### Set Instrument and Peak-Profile Parameters

These values provide the initial instrument description for the local
refinement. Later, a subset of them will be refined.

In [12]:
experiment.instrument.setup_wavelength = 1.494
experiment.instrument.calib_twotheta_offset = 0.0

In [13]:
experiment.peak.broad_gauss_u = 0.1
experiment.peak.broad_gauss_v = -0.1
experiment.peak.broad_gauss_w = 0.1204
experiment.peak.broad_lorentz_y = 0.0844

#### Add Background Points and Excluded Regions

The line-segment background is defined by a few anchor points. We also
exclude regions that are not intended to contribute to the fit.

In [14]:
experiment.background.create(id='1', x=10, y=168.5585)
experiment.background.create(id='2', x=30, y=164.3357)
experiment.background.create(id='3', x=50, y=166.8881)
experiment.background.create(id='4', x=110, y=175.4006)

In [15]:
experiment.excluded_regions.create(id='1', start=0, end=10)
experiment.excluded_regions.create(id='2', start=100, end=180)

#### Link the Structural Phase to the Experiment

In [16]:
experiment.linked_phases.create(id='lbco', scale=9.1351)

## Step 4: Run an Initial Local Refinement

Before Bayesian sampling, it is useful to run a deterministic fit. This
gives us:

- a good point estimate near the best-fit region,
- uncertainties from the local optimizer,
- a quick check that the model and experiment are configured
  sensibly.

In this tutorial we refine only a small set of parameters that are easy
to interpret in the later Bayesian stage.

In [17]:
structure.cell.length_a.free = True

In [18]:
experiment.linked_phases['lbco'].scale.free = True
experiment.peak.broad_gauss_u.free = True
experiment.peak.broad_gauss_v.free = True
experiment.instrument.calib_twotheta_offset.free = True

We choose the BUMPS Levenberg-Marquardt minimizer as a fast local
optimizer. Its main purpose here is to provide a stable starting point
and uncertainty estimates for the Bayesian run.

In [19]:
project.analysis.fit.show_minimizer_types()

Minimizer types


,,Type,Description
1,,bumps,Bumps library using the default Levenberg-Marquardt method
2,,bumps (amoeba),Bumps library with Nelder-Mead simplex method
3,,bumps (de),Bumps library with differential evolution method
4,,bumps (dream),Bumps library with DREAM Bayesian sampling
5,,bumps (lm),Bumps library with Levenberg-Marquardt method
6,,dfols,DFO-LS library for derivative-free least-squares optimization
7,,lmfit,LMFIT library using the default Levenberg-Marquardt least squares method
8,,lmfit (least_squares),LMFIT library with SciPy's trust region reflective algorithm
9,*,lmfit (leastsq),LMFIT library with Levenberg-Marquardt least squares method


In [20]:
project.analysis.fit.minimizer_type = 'bumps (lm)'

Current minimizer changed to
bumps (lm)


In [21]:
project.analysis.fit()

Standard fitting
📋 Using experiment 🔬 'hrpt' for 'single' fitting
🚀 Starting fit process with 'bumps (lm)'...
📈 Goodness-of-fit progress:


,iteration,time (s),χ²,change / status
1,1,0.19,377.51,
2,8,0.37,56.31,85.1% ↓
3,14,0.50,39.54,29.8% ↓
4,21,0.66,37.66,4.7% ↓
5,26,0.77,34.14,9.4% ↓
6,27,0.80,23.47,31.3% ↓
7,33,0.93,8.74,62.7% ↓
8,39,1.06,1.85,78.9% ↓
9,45,1.19,1.30,29.9% ↓
10,77,1.87,1.29,


🏆 Best goodness-of-fit (reduced χ²) is 1.29 at iteration 76
✅ Fitting complete.


In [22]:
project.analysis.display.fit_results()

Fit results
✅ Success: True
⏱️ Fitting time: 1.87 seconds
📏 Goodness-of-fit (reduced χ²): 1.29
📏 R-factor (Rf): 5.65%
📏 R-factor squared (Rf²): 4.92%
📏 Weighted R-factor (wR): 4.08%
📈 Fitted parameters:


,datablock,category,entry,parameter,start,fitted,uncertainty,units,change
1,lbco,cell,,length_a,3.8800,3.8913,0.0001,Å,0.29 % ↑
2,hrpt,linked_phases,lbco,scale,9.1351,9.1329,0.0333,,0.02 % ↓
3,hrpt,peak,,broad_gauss_u,0.1000,0.0817,0.0078,deg²,18.33 % ↓
4,hrpt,peak,,broad_gauss_v,-0.1000,-0.1169,0.0057,deg²,16.91 % ↑
5,hrpt,instrument,,twotheta_offset,0.0000,0.6306,0.0019,deg,N/A


The correlation plot shows how strongly the fitted parameters move
together in the local refinement. The measured-vs-calculated plots show
how well the refined model reproduces the data globally and in a zoomed
region.

In [23]:
project.display.plotter.plot_param_correlations()

In [24]:
project.display.plotter.plot_meas_vs_calc(expt_name='hrpt')

In [25]:
project.display.plotter.plot_meas_vs_calc(expt_name='hrpt', x_min=65, x_max=68)

## Step 5: Prepare for Bayesian Sampling

DREAM requires finite bounds for the free parameters. Instead of
setting them manually, we derive them from the uncertainties estimated
in the local refinement.

The helper method `set_fit_bounds_from_uncertainty` centers the bounds
on the current parameter value and expands them by a chosen multiple of
the reported uncertainty.

Default multiplier is 8 to give a wide range for the sampler to
explore, but here we use 3 to speed up the tutorial.

In [26]:
project.analysis.display.free_params()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,lbco,cell,,length_a,3.89134,0.00011,-inf,inf,Å
2,hrpt,linked_phases,lbco,scale,9.13288,0.03329,-inf,inf,
3,hrpt,peak,,broad_gauss_u,0.08167,0.00783,-inf,inf,deg²
4,hrpt,peak,,broad_gauss_v,-0.11691,0.00566,-inf,inf,deg²
5,hrpt,instrument,,twotheta_offset,0.63057,0.00191,-inf,inf,deg


In [27]:
for param in project.free_parameters:
    param.set_fit_bounds_from_uncertainty(multiplier=3)

Displaying the free parameters again is a convenient way to confirm
that the fit bounds have been assigned as expected before launching the
sampler.

In [28]:
project.analysis.display.free_params()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,lbco,cell,,length_a,3.89134,0.00011,3.89101,3.89166,Å
2,hrpt,linked_phases,lbco,scale,9.13288,0.03329,9.03302,9.23274,
3,hrpt,peak,,broad_gauss_u,0.08167,0.00783,0.05817,0.10517,deg²
4,hrpt,peak,,broad_gauss_v,-0.11691,0.00566,-0.13389,-0.09993,deg²
5,hrpt,instrument,,twotheta_offset,0.63057,0.00191,0.62483,0.63631,deg


## Step 6: Configure and Run DREAM

We now switch from the local minimizer to the Bayesian DREAM sampler.

The settings below are intentionally small so the tutorial runs
quickly. For production analysis you would usually increase the number
of steps (`steps`) and often the burn-in (`burn`) as well. When
needed, the DREAM API also lets you tune how chains are initialized
through the `init` setting. Other sampler settings such as `thin` and
`pop` can be adjusted  as well, but here we keep them at their
defaults.

In [29]:
project.analysis.fit.show_minimizer_types()

Minimizer types


,,Type,Description
1,,bumps,Bumps library using the default Levenberg-Marquardt method
2,,bumps (amoeba),Bumps library with Nelder-Mead simplex method
3,,bumps (de),Bumps library with differential evolution method
4,,bumps (dream),Bumps library with DREAM Bayesian sampling
5,*,bumps (lm),Bumps library with Levenberg-Marquardt method
6,,dfols,DFO-LS library for derivative-free least-squares optimization
7,,lmfit,LMFIT library using the default Levenberg-Marquardt least squares method
8,,lmfit (least_squares),LMFIT library with SciPy's trust region reflective algorithm
9,,lmfit (leastsq),LMFIT library with Levenberg-Marquardt least squares method


In [30]:
project.analysis.fit.minimizer_type = 'bumps (dream)'

Current minimizer changed to
bumps (dream)


In [31]:
project.analysis.fit.minimizer.steps = 100  # 1000

In [32]:
project.analysis.fit()

Standard fitting
📋 Using experiment 🔬 'hrpt' for 'single' fitting
🚀 Starting fit process with 'bumps (dream)'...
📈 Bayesian sampling progress:


,iteration,progress,time (s),log posterior,phase
1,1/151,0.7%,0.43,-1157.01,burn-in
2,13/151,8.6%,5.49,-1157.54,burn-in
3,26/151,17.2%,10.82,-1158.21,burn-in
4,38/151,25.2%,15.85,-1159.61,burn-in
5,50/151,33.1%,20.89,-1159.52,burn-in
6,51/151,33.8%,21.36,-1159.49,sampling
7,56/151,37.1%,23.41,-1159.26,sampling
8,62/151,41.1%,25.92,-1159.52,sampling
9,67/151,44.4%,27.96,-1159.24,sampling
10,72/151,47.7%,30.23,-1159.00,sampling


⚠️ DREAM sampling completed, but convergence diagnostics indicate the posterior may be poorly mixed.                              
✅ Bayesian sampling complete.


## Step 7: Inspect Bayesian Results

The fit-results display now includes sampler settings, convergence
diagnostics, committed parameter values, and posterior summary
statistics.

In [33]:
project.analysis.display.fit_results()

Bayesian fit results
✅ Success: True
i Status: DREAM sampling completed
🧪 Sampler: dream
🎯 Committed point estimate: Max posterior
🔁 Sampler completed: True
⏱️ Fitting time: 63.47 seconds
📏 Goodness-of-fit (reduced χ²): 1.29
📉 Best log-posterior: -1157.00
⚙️ Sampler settings: random_seed=1708713996, steps=100, burn=50, thin=1, pop=4, samples=2000
📊 Convergence: converged=failed, max_r_hat=1.291, min_ess_bulk=56.4, draws=100, chains=20
📏 R-factor (Rf): 5.65%
📏 R-factor squared (Rf²): 4.92%
📏 Weighted R-factor (wR): 4.09%
📈 Committed parameters:


,datablock,category,entry,parameter,start,max posterior,uncertainty,units,change
1,lbco,cell,,length_a,3.8913,3.8913,0.0001,Å,0.00 % ↓
2,hrpt,linked_phases,lbco,scale,9.1329,9.1336,0.0305,,0.01 % ↑
3,hrpt,peak,,broad_gauss_u,0.0817,0.0817,0.0060,deg²,0.01 % ↓
4,hrpt,peak,,broad_gauss_v,-0.1169,-0.1169,0.0044,deg²,0.01 % ↑
5,hrpt,instrument,,twotheta_offset,0.6306,0.6303,0.0017,deg,0.05 % ↓


📊 Posterior parameter summaries:


,datablock,category,entry,parameter,median,std,68% interval,95% interval,r_hat,ess_bulk,units
1,lbco,cell,,length_a,3.8913,0.0001,"[3.8912, 3.8914]","[3.8912, 3.8915]",1.291,56.4,Å
2,hrpt,linked_phases,lbco,scale,9.1341,0.0305,"[9.1029, 9.1650]","[9.0758, 9.1901]",1.208,78.6,
3,hrpt,peak,,broad_gauss_u,0.0816,0.0060,"[0.0750, 0.0872]","[0.0684, 0.0928]",1.230,72.5,deg²
4,hrpt,peak,,broad_gauss_v,-0.1167,0.0044,"[-0.1211, -0.1126]","[-0.1251, -0.1076]",1.226,72.6,deg²
5,hrpt,instrument,,twotheta_offset,0.6304,0.0017,"[0.6286, 0.6320]","[0.6271, 0.6337]",1.281,58.6,deg


⚠️ r_hat: exceeds 1.01 (consider longer sampling, tighter bounds, or reparameterization).                                         
⚠️ ess_bulk: less than 400 (consider longer sampling, tighter bounds, or reparameterization).                                     


The correlation and posterior-pair plots are complementary:

- `plot_param_correlations` summarizes pairwise structure in a compact
  matrix.
- `plot_posterior_pairs` shows marginal densities on the diagonal and
  posterior contours off-diagonal.

In [34]:
project.display.plotter.plot_param_correlations()

In [35]:
project.display.plotter.plot_posterior_pairs()

The one-dimensional posterior distributions below make it easier to
inspect individual parameters in isolation, including asymmetry or
multimodality.

In [36]:
for param in project.free_parameters:
    project.display.plotter.plot_param_distribution(param)

Finally, the posterior predictive plot propagates the sampled parameter
uncertainty into the calculated diffraction pattern. Comparing this to
the zoomed measured-vs-calculated view helps assess whether the sampled
model family explains the data in the region of interest.

In [37]:
project.display.plotter.plot_posterior_predictive(expt_name='hrpt')

A final zoomed measured-vs-calculated plot is useful for checking how
the posterior-supported model behaves in a narrow region of the pattern
after the Bayesian run.

In [47]:
project.display.plotter.plot_posterior_predictive(expt_name='hrpt', x_min=45.4, x_max=46.3)